# 00 · Prepare one city (template)

Copy this notebook to start a model notebook. Change the two values in the first code cell, run the
second, and you hold the same frames as everyone else who names the same city and snapshot
([`docs/pipeline.md`](../docs/pipeline.md)). Everything below the second cell is yours to change.

Before the first run, download the snapshot (from the repo root):

```bash
python scripts/download_data.py toronto@2026-06-15
```


In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="Pandas requires version")  # Anaconda 'bottleneck' noise
# `pip install -e .` once makes the next line unnecessary; it only finds the repo root when the package is not installed
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "strfair").is_dir())))
import strfair

CITY = "toronto"           # one of strfair.PINNED_SNAPSHOTS
SNAPSHOT = "2026-06-15"    # the team's date for that city (required: there is no "latest")

In [2]:
data = strfair.prepare(CITY, SNAPSHOT)   # compare the `data` hash with a teammate: same hash, same data

toronto 2026-06-15 | listings 22,198 rows, 18,262 in sample | features 106 cols | code 025a7a1b | data 597d7dbb  (cached)


## The frames

One DataFrame per Inside Airbnb file. `listings` (with the `in_sample` flag) is the contract; `features`
is a starting point built from the in-sample rows, in the same row order as `data.sample`. The other
files load on first access.

In [3]:
print(f"listings {data.listings.shape}   sample {data.sample.shape}   features {data.features.shape}")
data.features.head(3)

listings (22198, 93)   sample (18262, 93)   features (18262, 106)


,id,city,snapshot,price,log_price,lead_days,quote_nights,weekend_checkin,checkin_dow,checkin_month,...,amg_family_pet,amg_exercise,amg_parking,amg_bathroom_laundry,amg_connectivity,am_wifi,am_free_parking,am_smoking_allowed,am_breakfast,price_per_person
0,27423,toronto,2026-06-15,16.90,2.827314,199,90,1.0,4,1,...,0,0,1,1,1,1,1,0,0,16.900
1,44452,toronto,2026-06-15,99.85,4.603669,46,28,1.0,5,8,...,0,0,1,1,1,1,1,0,0,49.925
2,44469,toronto,2026-06-15,76.70,4.339902,107,30,0.0,3,10,...,0,0,0,1,1,1,0,0,0,38.350


In [4]:
# which builder each standard feature comes from (strfair/features.py); add your own columns freely
strfair.feature_families(data.sample).value_counts()

family
reviews         20
target_quote    16
amenities       14
host            12
listing_text    12
availability    12
size            10
location         6
Name: count, dtype: int64

## From here on it is your notebook

Pick columns, split or cross-validate, impute, encode, fit. A few helpers you may want:

- `strfair.LEAKY`: columns that are functions of the price itself (targets, revenue, value score). Never inputs.
- `strfair.DEMAND_RESPONSE`: availability and recent-review columns. The proposal uses them to validate labels, not as inputs.
- `strfair.CATEGORICAL`: the categorical columns among the standard features.
- `strfair.impute_sizes(df, train=mask)`: median-by-room-type fill for `bedrooms` / `beds` / `baths` plus `_missing` flags.

Because every notebook holds the identical frame, `KFold(5, shuffle=True, random_state=42)` gives
everyone the same folds without any shared split column. Out-of-fold predictions give a residual for
every listing, which is what the fair-price labels need.

In [5]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

F = data.features
drop = strfair.LEAKY + strfair.DEMAND_RESPONSE + strfair.ID_COLUMNS + ["neighbourhood_raw"]  # raw = unpooled copy of `neighbourhood`
X_cols = [c for c in F.columns if c not in drop]
cat = list(F[X_cols].select_dtypes(exclude="number").columns)   # strfair.CATEGORICAL, plus anything you add
num = [c for c in X_cols if c not in cat]

pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), cat),
])
model = make_pipeline(pre, ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000))
cv = KFold(5, shuffle=True, random_state=42)
oof = cross_val_predict(model, F[X_cols], F["log_price"], cv=cv, n_jobs=-1)

print(f"{len(X_cols)} inputs | 5-fold out-of-fold MAE on log price: {mean_absolute_error(F['log_price'], oof):.3f}   "
      f"on dollars: ${mean_absolute_error(F['price'], np.exp(oof)):,.0f}")
F = F.assign(fair_log_price=oof, residual=F["log_price"] - oof)   # one residual per listing
F[["id", "price", "fair_log_price", "residual"]].head()

80 inputs | 5-fold out-of-fold MAE on log price: 0.276   on dollars: $104


,id,price,fair_log_price,residual
0,27423,16.90,3.586548,-0.759235
1,44452,99.85,4.607073,-0.003404
2,44469,76.70,4.551751,-0.211849
3,45399,107.41,4.869300,-0.192647
4,45893,71.86,4.409892,-0.135172


## The other files, when you need them

In [6]:
reviews = data.reviews               # (listing_id, date) of every review in the snapshot
print(f"{len(reviews):,} reviews, {reviews.in_sample.mean():.0%} of them on in-sample listings")
# data.reviews_text   -> adds the comments (100+ MB for big cities)
# data.calendar       -> 365 days x listings, availability only (11 M rows for NYC)
# data.neighbourhoods, data.geojson, data.listings_summary

679,139 reviews, 88% of them on in-sample listings
